# E-Commerce Customer Intelligence Analysis

**Objective:** Identify revenue concentration, customer value tiers, repeat-purchase behavior, promotion patterns, and product/category opportunities from customer shopping data.

**Workflow:** CSV → Data Quality → Cleaning → Feature Engineering → EDA → SQL Business Analysis → Power BI


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_shopping_behavior.csv")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")


In [ ]:
# Data-quality review
print(df.head())
print("\nShape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])


In [ ]:
# Cleaning
# Review ratings are the only missing analytical field in this dataset.
# Use category-level medians so the imputation respects category-specific rating patterns.
df["Review Rating"] = (
    df.groupby("Category")["Review Rating"]
      .transform(lambda s: s.fillna(s.median()))
)

# Standardize names for consistent Python and SQL usage.
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_")
)
df = df.rename(columns={"purchase_amount_usd": "purchase_amount"})

print("Remaining missing values:", int(df.isna().sum().sum()))


## Feature Engineering

In [ ]:
# 1) Business-friendly age bands
age_bins = [-1, 24, 34, 44, np.inf]
age_labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels)

# 2) Convert purchase cadence into an approximate number of days.
frequency_days = {
    "Weekly": 7, "Fortnightly": 14, "Bi-Weekly": 14,
    "Monthly": 30, "Every 3 Months": 90, "Quarterly": 90, "Annually": 365
}
df["purchase_frequency_days"] = df["frequency_of_purchases"].map(frequency_days)

# 3) Transparent behavioral segmentation using purchase history.
df["customer_segment"] = pd.cut(
    df["previous_purchases"],
    bins=[-np.inf, 5, 20, np.inf],
    labels=["New / Low-Repeat", "Returning", "Loyal"]
)

# 4) A separate value tier based on the current transaction amount.
# qcut creates four equally populated groups, avoiding arbitrary dollar cutoffs.
df["value_tier"] = pd.qcut(
    df["purchase_amount"],
    q=4,
    labels=["Entry", "Core", "Premium", "High Value"]
)

# 5) Combine promotion and subscription status into a business audience flag.
df["promotion_audience"] = np.select(
    [
        (df["discount_applied"] == "Yes") & (df["subscription_status"] == "Yes"),
        (df["discount_applied"] == "Yes") & (df["subscription_status"] == "No")
    ],
    ["Discounted Subscriber", "Discounted Non-Subscriber"],
    default="No Discount"
)

# Promo-code usage is highly redundant with discount status for this dataset.
if "promo_code_used" in df.columns:
    df = df.drop(columns=["promo_code_used"])

df[["customer_segment", "value_tier", "promotion_audience"]].head()


## Executive KPIs

In [ ]:
kpis = {
    "Total Revenue": df["purchase_amount"].sum(),
    "Transactions": len(df),
    "Average Order Value": df["purchase_amount"].mean(),
    "Unique Customers": df["customer_id"].nunique(),
    "Subscribed Customers": (df["subscription_status"] == "Yes").sum(),
    "Discounted Orders": (df["discount_applied"] == "Yes").sum()
}
pd.Series(kpis).round(2)


## Revenue Concentration

In [ ]:
category_summary = (
    df.groupby("category")
      .agg(revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           transactions=("customer_id", "count"))
      .sort_values("revenue", ascending=False)
)
category_summary["revenue_share_pct"] = category_summary["revenue"] / df["purchase_amount"].sum() * 100
category_summary.round(2)


In [ ]:
location_summary = (
    df.groupby("location")
      .agg(revenue=("purchase_amount", "sum"),
           transactions=("customer_id", "count"),
           avg_order_value=("purchase_amount", "mean"))
      .sort_values("revenue", ascending=False)
)
location_summary.head(10).round(2)


## Product & Rating Analysis

In [ ]:
item_summary = (
    df.groupby("item_purchased")
      .agg(revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           transactions=("customer_id", "count"),
           avg_rating=("review_rating", "mean"))
      .sort_values("revenue", ascending=False)
)
item_summary["revenue_share_pct"] = item_summary["revenue"] / df["purchase_amount"].sum() * 100
item_summary.head(10).round(2)


In [ ]:
# Identify products that combine commercial performance with strong customer ratings.
product_quality = item_summary.query("transactions >= 50").copy()
product_quality["rating_revenue_score"] = product_quality["avg_rating"] * product_quality["revenue_share_pct"]
product_quality.sort_values("rating_revenue_score", ascending=False).head(10).round(2)


## Customer Intelligence

In [ ]:
segment_summary = (
    df.groupby("customer_segment", observed=False)
      .agg(customers=("customer_id", "count"),
           revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           avg_previous_purchases=("previous_purchases", "mean"))
)
segment_summary["revenue_share_pct"] = segment_summary["revenue"] / df["purchase_amount"].sum() * 100
segment_summary.round(2)


In [ ]:
value_tier_summary = (
    df.groupby("value_tier", observed=False)
      .agg(customers=("customer_id", "count"),
           revenue=("purchase_amount", "sum"),
           avg_previous_purchases=("previous_purchases", "mean"))
)
value_tier_summary["revenue_share_pct"] = value_tier_summary["revenue"] / df["purchase_amount"].sum() * 100
value_tier_summary.round(2)


In [ ]:
subscription_summary = (
    df.groupby("subscription_status")
      .agg(customers=("customer_id", "count"),
           revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           avg_previous_purchases=("previous_purchases", "mean"))
)
subscription_summary["revenue_share_pct"] = subscription_summary["revenue"] / df["purchase_amount"].sum() * 100
subscription_summary.round(2)


## Promotion & Basket Behavior

In [ ]:
promotion_summary = (
    df.groupby("promotion_audience")
      .agg(orders=("customer_id", "count"),
           revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"))
      .sort_values("revenue", ascending=False)
)
promotion_summary.round(2)


In [ ]:
discount_by_item = (
    df.groupby("item_purchased")
      .agg(orders=("customer_id", "count"),
           discounted_orders=("discount_applied", lambda s: (s == "Yes").sum()))
)
discount_by_item["discount_rate_pct"] = discount_by_item["discounted_orders"] / discount_by_item["orders"] * 100
discount_by_item.sort_values("discount_rate_pct", ascending=False).head(10).round(2)


## Season, Shipping, Payment & Age

In [ ]:
season_summary = (
    df.groupby("season")
      .agg(revenue=("purchase_amount", "sum"), avg_order_value=("purchase_amount", "mean"), transactions=("customer_id", "count"))
      .sort_values("revenue", ascending=False)
)
shipping_summary = (
    df.groupby("shipping_type")
      .agg(revenue=("purchase_amount", "sum"), avg_order_value=("purchase_amount", "mean"), transactions=("customer_id", "count"))
      .sort_values("avg_order_value", ascending=False)
)
payment_summary = (
    df.groupby("payment_method")
      .agg(revenue=("purchase_amount", "sum"), avg_order_value=("purchase_amount", "mean"), transactions=("customer_id", "count"))
      .sort_values("revenue", ascending=False)
)
age_summary = (
    df.groupby("age_group", observed=False)
      .agg(revenue=("purchase_amount", "sum"), avg_order_value=("purchase_amount", "mean"), transactions=("customer_id", "count"))
)
print("Season\n", season_summary.round(2))
print("\nShipping\n", shipping_summary.round(2))
print("\nPayment\n", payment_summary.round(2))
print("\nAge\n", age_summary.round(2))


## Analyst Takeaways

Focus the final dashboard on four questions: **Where is revenue concentrated? Which customer/value tiers matter most? Where are promotions being used? Which products/categories combine scale with customer satisfaction?** Avoid claiming causal effects or true retention/churn because the dataset is cross-sectional and does not contain a full transaction history over time.
